In [ ]:
# Path to project_root, data_dir, model_dir, and out_dir must be configured before running.
# Path to top1000_path, qrels_path, queries_path, and collection_path is resolved under data_dir.
CFG = {
    "project_root": "<path/to/project_root>",
    "data_dir": "<path/to/data_dir>",
    "model_dir": "<path/to/monot5_model_dir>",
    "out_dir": "<path/to/output_dir>",

    "top1000_path": "top1000.dev",
    "qrels_path": "qrels.dev.small.tsv",
    "queries_path": "queries.dev.small.tsv",
    "collection_path": "collection.tsv",

    "max_input_length": 512,
    "batch_size": 8,
    "query_limit": None,
    "smoke_query_limit": 5,

    "use_fp16_if_cuda": True,
    "score_mode": "logprob_true_minus_false",
    "run_tag": "monoT5-base",
    "flush_every_queries": 25,

    "force_recompute": False,
}


In [ ]:
import os

os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["HF_DATASETS_OFFLINE"] = "1"

import json
import math
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

pd.set_option("display.max_colwidth", 180)
pd.set_option("display.width", 160)

print("Offline flags:")
print("  TRANSFORMERS_OFFLINE=", os.environ.get("TRANSFORMERS_OFFLINE"))
print("  HF_HUB_OFFLINE=", os.environ.get("HF_HUB_OFFLINE"))
print("  HF_DATASETS_OFFLINE=", os.environ.get("HF_DATASETS_OFFLINE"))


In [ ]:
def resolve_under(base, value):
    path = Path(value).expanduser()
    return path.resolve() if path.is_absolute() else (base / path).resolve()


PROJECT_ROOT = Path(CFG["project_root"]).expanduser().resolve()
DATA_DIR = resolve_under(PROJECT_ROOT, CFG["data_dir"])
MODEL_DIR = resolve_under(PROJECT_ROOT, CFG["model_dir"])
OUT_DIR = resolve_under(PROJECT_ROOT, CFG["out_dir"])
PER_QUERY_DIR = OUT_DIR / "per_query"

PATHS = {
    "project_root": PROJECT_ROOT,
    "data_dir": DATA_DIR,
    "model_dir": MODEL_DIR,
    "out_dir": OUT_DIR,
    "per_query_dir": PER_QUERY_DIR,
    "top1000": resolve_under(DATA_DIR, CFG["top1000_path"]),
    "qrels": resolve_under(DATA_DIR, CFG["qrels_path"]),
    "queries": resolve_under(DATA_DIR, CFG["queries_path"]),
    "collection": resolve_under(DATA_DIR, CFG["collection_path"]),
}

RUN_PATH = OUT_DIR / "monot5_base_full_eval.run"
METRICS_PATH = OUT_DIR / "monot5_base_full_eval.metrics.json"
CONFIG_PATH = OUT_DIR / "monot5_base_full_eval.config.json"
PROGRESS_PATH = OUT_DIR / "monot5_base_full_eval.progress.json"
MISSING_PATH = OUT_DIR / "monot5_base_full_eval.missing.json"
PREVIEW_PATH = OUT_DIR / "monot5_base_full_eval.preview.tsv"

OUT_DIR.mkdir(parents=True, exist_ok=True)
PER_QUERY_DIR.mkdir(parents=True, exist_ok=True)

print("Resolved paths:")
for key, value in PATHS.items():
    print(f"  {key}: {value}")
print("Output files:")
for path in [RUN_PATH, METRICS_PATH, CONFIG_PATH, PROGRESS_PATH, MISSING_PATH, PREVIEW_PATH]:
    print(f"  {path}")


In [ ]:
def split_nontext_line(line):
    line = line.rstrip("\n")
    return line.split("\t") if "\t" in line else line.split()


def read_first_nonempty_lines(path, n=5):
    lines = []
    with path.open("r", encoding="utf-8", errors="replace") as handle:
        for line in handle:
            if line.strip():
                lines.append(line.rstrip("\n"))
            if len(lines) >= n:
                break
    return lines


def looks_like_int(value):
    try:
        int(str(value))
        return True
    except Exception:
        return False


def looks_like_float(value):
    try:
        float(str(value))
        return True
    except Exception:
        return False


def looks_like_text(value):
    value = str(value)
    has_alpha = any(ch.isalpha() for ch in value)
    return has_alpha and (" " in value or len(value) >= 20)


def detect_top1000_format(path):
    lines = read_first_nonempty_lines(path, n=10)
    if not lines:
        raise ValueError(f"No non-empty rows found in top1000 file: {path}")

    tab_parts = [line.split("\t") for line in lines]
    generic_parts = [split_nontext_line(line) for line in lines]

    text_like = [
        len(parts) >= 4
        and str(parts[1]).upper() != "Q0"
        and looks_like_text(parts[2])
        and looks_like_text(parts[3])
        for parts in tab_parts
    ]
    if all(text_like):
        return {"kind": "text", "description": "qid, pid, query_text, passage_text"}

    trec_like = [
        len(parts) >= 6
        and str(parts[1]).upper() == "Q0"
        and looks_like_int(parts[3])
        and looks_like_float(parts[4])
        for parts in generic_parts
    ]
    if all(trec_like):
        return {"kind": "trec_run", "description": "qid, Q0, pid, rank, score, tag"}

    id_like = []
    for parts in generic_parts:
        if len(parts) < 2:
            id_like.append(False)
            continue
        if len(parts) >= 4 and looks_like_text(parts[2]) and looks_like_text(parts[3]):
            id_like.append(False)
            continue
        id_like.append(True)
    if all(id_like):
        return {"kind": "id_run", "description": "ID-only candidate file; query/passage text will be joined from local TSV files"}

    preview = "\n".join(lines[:3])
    raise ValueError(
        "Could not infer top1000.dev format. Expected either:\n"
        "  A) qid<TAB>pid<TAB>query_text<TAB>passage_text\n"
        "  B) qid Q0 pid rank score tag, or another ID-only qid/pid format\n"
        f"Preview:\n{preview}"
    )

TOP1000_FORMAT = detect_top1000_format(PATHS["top1000"])
print("Detected top1000 format:", TOP1000_FORMAT)
print("Preview:")
for line in read_first_nonempty_lines(PATHS["top1000"], n=5):
    print(line[:500])


In [ ]:
def qid_sort_key(qid):
    s = str(qid)
    return (0, int(s)) if s.isdigit() else (1, s)


def pid_sort_key(pid):
    s = str(pid)
    return (0, int(s)) if s.isdigit() else (1, s)


def parse_int(value, default=None):
    try:
        return int(str(value))
    except Exception:
        return default


def parse_float(value, default=None):
    try:
        return float(str(value))
    except Exception:
        return default


def atomic_write_text(path, text):
    path = Path(path)
    tmp = path.with_suffix(path.suffix + ".tmp")
    tmp.write_text(text, encoding="utf-8")
    tmp.replace(path)


def save_json(obj, path):
    atomic_write_text(path, json.dumps(obj, indent=2, ensure_ascii=False, sort_keys=True))


def load_qrels(path):
    qrels = {}
    raw_rows = 0
    positive_rows = 0
    with Path(path).open("r", encoding="utf-8", errors="replace") as handle:
        for line in handle:
            if not line.strip():
                continue
            raw_rows += 1
            parts = line.rstrip("\n").split("\t")
            if len(parts) >= 4:
                qid, pid, rel = parts[0], parts[2], parts[3]
            elif len(parts) >= 3:
                qid, pid, rel = parts[0], parts[1], parts[2]
            else:
                raise ValueError(f"Malformed qrels row {raw_rows}: {line[:200]!r}")
            rel_value = parse_int(rel, default=None)
            if rel_value is None:
                rel_value = int(float(rel))
            if rel_value > 0:
                qrels.setdefault(str(qid), {})[str(pid)] = int(rel_value)
                positive_rows += 1
    if not qrels:
        raise ValueError(f"No positive qrels found in {path}")
    return qrels, {"raw_rows": raw_rows, "positive_rows": positive_rows, "num_qids": len(qrels)}


def load_queries(path):
    queries = {}
    with Path(path).open("r", encoding="utf-8", errors="replace") as handle:
        for line_no, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            parts = line.rstrip("\n").split("\t", 1)
            if len(parts) != 2:
                raise ValueError(f"Malformed query row {line_no}: {line[:200]!r}")
            queries[str(parts[0])] = parts[1]
    if not queries:
        raise ValueError(f"No queries loaded from {path}")
    return queries


def load_collection_subset(path, needed_pids):
    needed = {str(pid) for pid in needed_pids}
    passages = {}
    if not needed:
        return passages
    with Path(path).open("r", encoding="utf-8", errors="replace") as handle:
        for line in tqdm(handle, desc="Loading required collection passages", unit=" lines"):
            if not line.strip():
                continue
            parts = line.rstrip("\n").split("\t", 1)
            if len(parts) != 2:
                continue
            pid, passage = str(parts[0]), parts[1]
            if pid in needed:
                passages[pid] = passage
                if len(passages) == len(needed):
                    break
    return passages


def load_text_top1000(path):
    df = pd.read_csv(
        path,
        sep="\t",
        header=None,
        usecols=[0, 1, 2, 3],
        names=["qid", "pid", "query", "passage"],
        dtype=str,
        keep_default_na=False,
    )
    df["qid"] = df["qid"].astype(str)
    df["pid"] = df["pid"].astype(str)
    df["orig_rank"] = df.groupby("qid", sort=False).cumcount() + 1
    df["source_format"] = "text"
    return df


def load_run_top1000(path, detected_format):
    records = []
    with Path(path).open("r", encoding="utf-8", errors="replace") as handle:
        for line_no, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            parts = split_nontext_line(line)
            kind = detected_format["kind"]
            if kind == "trec_run":
                if len(parts) < 6:
                    raise ValueError(f"Malformed TREC run row {line_no}: {line[:200]!r}")
                qid, pid = parts[0], parts[2]
                orig_rank = parse_int(parts[3], default=None)
                candidate_score = parse_float(parts[4], default=np.nan)
            else:
                if len(parts) < 2:
                    raise ValueError(f"Malformed ID-only row {line_no}: {line[:200]!r}")
                qid, pid = parts[0], parts[1]
                orig_rank = parse_int(parts[2], default=None) if len(parts) >= 3 else None
                candidate_score = parse_float(parts[3], default=np.nan) if len(parts) >= 4 else np.nan
            records.append({
                "qid": str(qid),
                "pid": str(pid),
                "orig_rank": orig_rank,
                "candidate_score": candidate_score,
                "input_order": len(records) + 1,
                "source_format": kind,
            })
    if not records:
        raise ValueError(f"No candidate rows loaded from {path}")

    df = pd.DataFrame.from_records(records)
    missing_rank = df["orig_rank"].isna()
    if missing_rank.any():
        df.loc[missing_rank, "orig_rank"] = df[missing_rank].groupby("qid", sort=False).cumcount() + 1
    df["orig_rank"] = df["orig_rank"].astype(int)
    return df


def load_candidates(top1000_path, detected_format, queries_path, collection_path):
    if detected_format["kind"] == "text":
        df = load_text_top1000(top1000_path)
        missing_queries = int((df["query"].astype(str).str.len() == 0).sum())
        missing_passages = int((df["passage"].astype(str).str.len() == 0).sum())
        missing = {
            "missing_query_rows": missing_queries,
            "missing_passage_rows": missing_passages,
            "missing_query_qids": sorted(df.loc[df["query"].astype(str).str.len() == 0, "qid"].unique().tolist(), key=qid_sort_key),
            "missing_passage_pids_sample": sorted(df.loc[df["passage"].astype(str).str.len() == 0, "pid"].unique().tolist(), key=pid_sort_key)[:50],
        }
        return df, missing

    df = load_run_top1000(top1000_path, detected_format)
    queries = load_queries(queries_path)
    needed_pids = df["pid"].astype(str).unique().tolist()
    passages = load_collection_subset(collection_path, needed_pids)

    df["query"] = df["qid"].map(queries)
    df["passage"] = df["pid"].map(passages)
    missing_query_mask = df["query"].isna()
    missing_passage_mask = df["passage"].isna()
    missing = {
        "missing_query_rows": int(missing_query_mask.sum()),
        "missing_passage_rows": int(missing_passage_mask.sum()),
        "missing_query_qids": sorted(df.loc[missing_query_mask, "qid"].unique().tolist(), key=qid_sort_key),
        "missing_passage_pids_sample": sorted(df.loc[missing_passage_mask, "pid"].unique().tolist(), key=pid_sort_key)[:50],
        "num_loaded_queries": len(queries),
        "num_loaded_passages_for_candidates": len(passages),
        "num_needed_candidate_pids": len(needed_pids),
    }
    return df, missing


def candidate_data_summary(candidates, qrels, missing):
    counts = candidates.groupby("qid").size()
    candidate_qids = set(candidates["qid"].astype(str).unique())
    qrels_qids = set(qrels.keys())
    overlap_qids = candidate_qids & qrels_qids
    return {
        "num_candidate_rows": int(len(candidates)),
        "num_candidate_qids": int(len(candidate_qids)),
        "num_qrels_qids": int(len(qrels_qids)),
        "num_qids_with_candidates_and_qrels": int(len(overlap_qids)),
        "num_missing_queries": int(missing.get("missing_query_rows", 0)),
        "num_missing_passages": int(missing.get("missing_passage_rows", 0)),
        "avg_candidates_per_query": float(counts.mean()) if len(counts) else 0.0,
        "min_candidates_per_query": int(counts.min()) if len(counts) else 0,
        "max_candidates_per_query": int(counts.max()) if len(counts) else 0,
    }


In [ ]:
qrels, qrels_info = load_qrels(PATHS["qrels"])
candidates, missing_info = load_candidates(PATHS["top1000"], TOP1000_FORMAT, PATHS["queries"], PATHS["collection"])

if len(candidates) == 0:
    raise ValueError("At least one candidate row is required.")

scoreable_candidates = candidates[
    candidates["query"].notna()
    & candidates["passage"].notna()
    & (candidates["query"].astype(str).str.len() > 0)
    & (candidates["passage"].astype(str).str.len() > 0)
].copy()

candidate_qids = set(candidates["qid"].astype(str).unique())
scoreable_qids = set(scoreable_candidates["qid"].astype(str).unique())
qrels_qids = set(qrels.keys())
evaluable_qids_all = sorted(scoreable_qids & qrels_qids, key=qid_sort_key)
if not evaluable_qids_all:
    raise ValueError(
        "No overlapping qids between scoreable candidates and qrels. "
        f"candidate_qids={len(candidate_qids)}, scoreable_qids={len(scoreable_qids)}, qrels_qids={len(qrels_qids)}"
    )

DATA_SUMMARY = candidate_data_summary(candidates, qrels, missing_info)
DATA_SUMMARY["num_scoreable_candidate_rows"] = int(len(scoreable_candidates))
DATA_SUMMARY["num_scoreable_candidate_qids"] = int(len(scoreable_qids))

save_json(missing_info, MISSING_PATH)

print("Qrels info:", qrels_info)
print("Data summary:")
print(json.dumps(DATA_SUMMARY, indent=2, ensure_ascii=False))
print("Missing info saved to:", MISSING_PATH)
print("Candidate sample:")
print(candidates.head(3)[["qid", "pid", "orig_rank", "query", "passage"]].to_string(index=False))


In [ ]:
def ranked_pids_from_rows(rows):
    if isinstance(rows, pd.DataFrame):
        return rows.sort_values("rank")["pid"].astype(str).tolist()
    if rows and isinstance(rows[0], dict):
        return [str(row["pid"]) for row in sorted(rows, key=lambda row: row["rank"])]
    return [str(pid) for pid in rows]


def dcg_at_k(ranked_pids, rels, k=10):
    total = 0.0
    for rank, pid in enumerate(ranked_pids[:k], start=1):
        rel = rels.get(str(pid), 0)
        if rel > 0:
            total += (2 ** rel - 1) / math.log2(rank + 1)
    return total


def idcg_at_k(rels, k=10):
    ideal = sorted([rel for rel in rels.values() if rel > 0], reverse=True)[:k]
    total = 0.0
    for rank, rel in enumerate(ideal, start=1):
        total += (2 ** rel - 1) / math.log2(rank + 1)
    return total


def compute_metrics(run_by_qid, qrels, qids=None, k=10):
    if qids is None:
        qids = sorted(set(run_by_qid.keys()) & set(qrels.keys()), key=qid_sort_key)
    else:
        qids = [str(qid) for qid in qids if str(qid) in run_by_qid and str(qid) in qrels]

    if not qids:
        raise ValueError("No qids available for metric computation.")

    rr_values = []
    hit_values = {1: [], 3: [], 5: [], 10: []}
    ndcg_values = []

    for qid in qids:
        ranked_pids = ranked_pids_from_rows(run_by_qid[qid])
        rels = qrels[qid]
        rel_pid_set = {pid for pid, rel in rels.items() if rel > 0}

        rr = 0.0
        for rank, pid in enumerate(ranked_pids[:k], start=1):
            if str(pid) in rel_pid_set:
                rr = 1.0 / rank
                break
        rr_values.append(rr)

        for cutoff in hit_values:
            hit_values[cutoff].append(float(any(str(pid) in rel_pid_set for pid in ranked_pids[:cutoff])))

        dcg = dcg_at_k(ranked_pids, rels, k=k)
        idcg = idcg_at_k(rels, k=k)
        ndcg_values.append(0.0 if idcg == 0.0 else dcg / idcg)

    return {
        "mrr@10": float(np.mean(rr_values)),
        "hit@1": float(np.mean(hit_values[1])),
        "hit@3": float(np.mean(hit_values[3])),
        "hit@5": float(np.mean(hit_values[5])),
        "hit@10": float(np.mean(hit_values[10])),
        "ndcg@10": float(np.mean(ndcg_values)),
        "num_qids": int(len(qids)),
    }


def candidate_order_run_by_qid(candidates_df, qids=None):
    source = candidates_df.copy()
    if qids is not None:
        wanted = {str(qid) for qid in qids}
        source = source[source["qid"].astype(str).isin(wanted)].copy()
    run_by_qid = {}
    for qid, group in source.groupby("qid", sort=False):
        ordered = group.sort_values(["orig_rank", "pid"])
        run_by_qid[str(qid)] = ordered["pid"].astype(str).tolist()
    return run_by_qid


In [ ]:
baseline_qids = sorted(set(candidates["qid"].astype(str).unique()) & set(qrels.keys()), key=qid_sort_key)
BM25_RUN_BY_QID = candidate_order_run_by_qid(candidates, baseline_qids)
BM25_METRICS = compute_metrics(BM25_RUN_BY_QID, qrels, baseline_qids, k=10)

print(f"Evaluated BM25 candidate-order baseline on {len(baseline_qids)} qids.")
print(json.dumps(BM25_METRICS, indent=2))


In [ ]:
def detect_device():
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = detect_device()
USE_CUDA_FP16 = DEVICE.type == "cuda" and bool(CFG["use_fp16_if_cuda"])
MODEL_DTYPE = torch.float16 if USE_CUDA_FP16 else torch.float32

print("Device:", DEVICE)
print("Model dtype:", MODEL_DTYPE)
if DEVICE.type in {"cpu", "mps"}:
    print("Warning: full monoT5 evaluation on CPU or MPS may be very slow. Use smoke/partial mode first.")

tokenizer = AutoTokenizer.from_pretrained(str(MODEL_DIR), local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(
    str(MODEL_DIR),
    local_files_only=True,
    torch_dtype=MODEL_DTYPE if DEVICE.type == "cuda" else None,
)
model.to(DEVICE)
model.eval()

DECODER_START_TOKEN_ID = model.config.decoder_start_token_id
if DECODER_START_TOKEN_ID is None:
    DECODER_START_TOKEN_ID = model.config.pad_token_id
if DECODER_START_TOKEN_ID is None:
    raise ValueError("Could not determine decoder start token id or pad token id for T5 scoring.")

print("Loaded local monoT5 model from:", MODEL_DIR)
print("decoder_start_token_id:", DECODER_START_TOKEN_ID)


In [ ]:
def get_single_token_id(label):
    candidates_to_try = [label, " " + label]
    attempts = []
    for text in candidates_to_try:
        ids = tokenizer.encode(text, add_special_tokens=False)
        tokens = tokenizer.convert_ids_to_tokens(ids)
        attempts.append({"text": text, "ids": ids, "tokens": tokens})
        if len(ids) == 1:
            return ids[0], attempts
    details = json.dumps(attempts, indent=2, ensure_ascii=False)
    raise ValueError(
        f"Label {label!r} did not tokenize to a single token. "
        "monoT5 first-token scoring needs single-token true/false labels. Attempts:\n"
        f"{details}"
    )

TRUE_TOKEN_ID, true_attempts = get_single_token_id("true")
FALSE_TOKEN_ID, false_attempts = get_single_token_id("false")

print("Tokenization for true:", true_attempts)
print("Tokenization for false:", false_attempts)
print("TRUE_TOKEN_ID:", TRUE_TOKEN_ID)
print("FALSE_TOKEN_ID:", FALSE_TOKEN_ID)


def build_monot5_prompt(query, passage):
    return f"Query: {query} Document: {passage} Relevant:"


def score_batch(query_texts, passage_texts):
    if len(query_texts) != len(passage_texts):
        raise ValueError("query_texts and passage_texts must have the same length")
    if not query_texts:
        return []

    prompts = [build_monot5_prompt(query, passage) for query, passage in zip(query_texts, passage_texts)]
    encoded = tokenizer(
        prompts,
        padding=True,
        truncation=True,
        max_length=int(CFG["max_input_length"]),
        return_tensors="pt",
    )
    encoded = {key: value.to(DEVICE) for key, value in encoded.items()}
    decoder_input_ids = torch.full(
        (len(prompts), 1),
        int(DECODER_START_TOKEN_ID),
        dtype=torch.long,
        device=DEVICE,
    )

    with torch.inference_mode():
        if USE_CUDA_FP16:
            with torch.autocast(device_type="cuda", dtype=torch.float16):
                outputs = model(**encoded, decoder_input_ids=decoder_input_ids)
        else:
            outputs = model(**encoded, decoder_input_ids=decoder_input_ids)

        first_logits = outputs.logits[:, 0, :]
        true_logits = first_logits[:, TRUE_TOKEN_ID]
        false_logits = first_logits[:, FALSE_TOKEN_ID]

        if CFG["score_mode"] == "logprob_true_minus_false":
            scores = true_logits - false_logits
        elif CFG["score_mode"] == "logprob_true":
            two_logits = torch.stack([false_logits, true_logits], dim=1)
            scores = torch.log_softmax(two_logits, dim=1)[:, 1]
        else:
            raise ValueError(f"Unsupported score_mode: {CFG['score_mode']}")

    return scores.detach().float().cpu().numpy().tolist()


def rerank_one_query(qid, query_candidates):
    qid = str(qid)
    group = query_candidates.copy().sort_values(["orig_rank", "pid"])
    records = group[["qid", "pid", "orig_rank", "query", "passage"]].to_dict("records")
    if not records:
        return []

    scores = []
    batch_size = int(CFG["batch_size"])
    for start in range(0, len(records), batch_size):
        batch = records[start:start + batch_size]
        scores.extend(score_batch([row["query"] for row in batch], [row["passage"] for row in batch]))

    for row, score in zip(records, scores):
        row["score"] = float(score)

    ranked = sorted(
        records,
        key=lambda row: (-row["score"], int(row["orig_rank"]), pid_sort_key(row["pid"])),
    )
    for rank, row in enumerate(ranked, start=1):
        row["rank"] = rank
        row["qid"] = qid
        row["pid"] = str(row["pid"])
        row["orig_rank"] = int(row["orig_rank"])
    return ranked


def trec_run_line(row):
    return f"{row['qid']} Q0 {row['pid']} {int(row['rank'])} {float(row['score']):.10f} {CFG['run_tag']}"


def verify_trec_run_lines(rows):
    for row in rows:
        parts = trec_run_line(row).split()
        if len(parts) != 6:
            raise ValueError(f"Invalid TREC line: {trec_run_line(row)!r}")
        int(parts[3])
        float(parts[4])
    return True


In [ ]:
smoke_qids = evaluable_qids_all[: int(CFG["smoke_query_limit"])]
if not smoke_qids:
    raise ValueError("No smoke-test qids available.")

smoke_run_by_qid = {}
smoke_rows = []
smoke_start = time.time()

for qid in tqdm(smoke_qids, desc="Smoke monoT5 scoring", unit="qid"):
    group = scoreable_candidates[scoreable_candidates["qid"].astype(str) == str(qid)]
    ranked = rerank_one_query(qid, group)
    verify_trec_run_lines(ranked)
    smoke_run_by_qid[str(qid)] = ranked
    smoke_rows.extend(ranked)

smoke_metrics = compute_metrics(smoke_run_by_qid, qrels, smoke_qids, k=10)
smoke_elapsed = time.time() - smoke_start

print(f"Smoke test elapsed: {smoke_elapsed:.2f} sec for {len(smoke_rows)} pairs")
print("Smoke metrics:")
print(json.dumps(smoke_metrics, indent=2))

for qid in smoke_qids:
    print("\nqid", qid)
    top5 = []
    rels = qrels.get(str(qid), {})
    for row in smoke_run_by_qid[str(qid)][:5]:
        top5.append({
            "rank": row["rank"],
            "pid": row["pid"],
            "score": row["score"],
            "orig_rank": row["orig_rank"],
            "is_relevant": int(rels.get(row["pid"], 0) > 0),
            "passage": str(row["passage"])[0:180],
        })
    print(pd.DataFrame(top5).to_string(index=False))

print("Run format example:")
print(trec_run_line(smoke_rows[0]))


In [ ]:
def qid_filename(qid):
    safe = "".join(ch if ch.isalnum() or ch in "._-" else "_" for ch in str(qid))
    return f"{safe}.json"


def per_query_path(qid):
    return PER_QUERY_DIR / qid_filename(qid)


def write_per_query_result(qid, ranked_rows):
    payload = {
        "qid": str(qid),
        "run_tag": CFG["run_tag"],
        "score_mode": CFG["score_mode"],
        "num_candidates": len(ranked_rows),
        "rows": [
            {
                "qid": str(row["qid"]),
                "pid": str(row["pid"]),
                "rank": int(row["rank"]),
                "score": float(row["score"]),
                "orig_rank": int(row["orig_rank"]),
            }
            for row in ranked_rows
        ],
    }
    save_json(payload, per_query_path(qid))


def read_per_query_result(qid):
    path = per_query_path(qid)
    payload = json.loads(path.read_text(encoding="utf-8"))
    rows = payload.get("rows", [])
    rows = sorted(rows, key=lambda row: int(row["rank"]))
    return rows


def save_progress(eval_qids, started_at, newly_scored_pairs):
    completed_qids = [str(qid) for qid in eval_qids if per_query_path(qid).exists()]
    payload = {
        "completed_qids": completed_qids,
        "num_completed_qids": len(completed_qids),
        "num_target_qids": len(eval_qids),
        "newly_scored_pairs_this_run": int(newly_scored_pairs),
        "elapsed_sec_this_run": float(time.time() - started_at),
        "updated_unix_time": float(time.time()),
        "per_query_dir": str(PER_QUERY_DIR),
        "run_path": str(RUN_PATH),
    }
    save_json(payload, PROGRESS_PATH)
    return payload


def merge_per_query_outputs(eval_qids, run_path):
    run_by_qid = {}
    lines = []
    total_rows = 0
    missing_qids = []
    for qid in eval_qids:
        path = per_query_path(qid)
        if not path.exists():
            missing_qids.append(str(qid))
            continue
        rows = read_per_query_result(qid)
        verify_trec_run_lines(rows)
        run_by_qid[str(qid)] = rows
        total_rows += len(rows)
        lines.extend(trec_run_line(row) for row in rows)
    if missing_qids:
        print(f"Warning: {len(missing_qids)} qids have no per-query output and will be absent from final run.")
    atomic_write_text(run_path, "\n".join(lines) + ("\n" if lines else ""))
    return run_by_qid, total_rows, missing_qids


def write_preview_tsv(run_by_qid, candidates_df, qrels, path, max_rows=50):
    preview_records = []
    text_source = candidates_df.drop_duplicates(["qid", "pid"], keep="first")
    text_lookup = text_source.set_index(["qid", "pid"])[["query", "passage"]].to_dict("index")
    for qid in sorted(run_by_qid.keys(), key=qid_sort_key):
        rels = qrels.get(str(qid), {})
        for row in run_by_qid[qid]:
            key = (str(qid), str(row["pid"]))
            text = text_lookup.get(key, {"query": "", "passage": ""})
            preview_records.append({
                "qid": str(qid),
                "pid": str(row["pid"]),
                "rank": int(row["rank"]),
                "score": float(row["score"]),
                "query": text.get("query", ""),
                "passage": text.get("passage", ""),
                "is_relevant": int(rels.get(str(row["pid"]), 0) > 0),
            })
            if len(preview_records) >= max_rows:
                pd.DataFrame(preview_records).to_csv(path, sep="\t", index=False)
                return
    pd.DataFrame(preview_records).to_csv(path, sep="\t", index=False)

query_limit = CFG["query_limit"]
eval_qids = list(evaluable_qids_all)
if query_limit is not None:
    eval_qids = eval_qids[: int(query_limit)]

if not eval_qids:
    raise ValueError("No qids selected for evaluation.")

print(f"Target qids: {len(eval_qids)}")
print(f"force_recompute: {CFG.get('force_recompute', False)}")
print(f"Per-query output dir: {PER_QUERY_DIR}")

started_at = time.time()
newly_scored_pairs = 0
newly_completed_qids = 0

pbar = tqdm(eval_qids, desc="Full monoT5 evaluation", unit="qid")
for index, qid in enumerate(pbar, start=1):
    qid_path = per_query_path(qid)
    if qid_path.exists() and not CFG.get("force_recompute", False):
        if index % int(CFG["flush_every_queries"]) == 0:
            save_progress(eval_qids, started_at, newly_scored_pairs)
        continue

    group = scoreable_candidates[scoreable_candidates["qid"].astype(str) == str(qid)]
    ranked = rerank_one_query(qid, group)
    if not ranked:
        continue
    verify_trec_run_lines(ranked)
    write_per_query_result(qid, ranked)

    newly_scored_pairs += len(ranked)
    newly_completed_qids += 1
    elapsed = time.time() - started_at
    pairs_per_sec = newly_scored_pairs / elapsed if elapsed > 0 else 0.0
    completed_total = sum(1 for item in eval_qids if per_query_path(item).exists())
    remaining_qids = max(0, len(eval_qids) - completed_total)
    avg_sec_per_new_qid = elapsed / newly_completed_qids if newly_completed_qids else 0.0
    eta_sec = remaining_qids * avg_sec_per_new_qid
    pbar.set_postfix({
        "new_pairs": newly_scored_pairs,
        "pairs/s": f"{pairs_per_sec:.2f}",
        "eta_min": f"{eta_sec / 60:.1f}",
        "out": RUN_PATH.name,
    })

    if newly_completed_qids % int(CFG["flush_every_queries"]) == 0:
        save_progress(eval_qids, started_at, newly_scored_pairs)

progress_payload = save_progress(eval_qids, started_at, newly_scored_pairs)
MONOT5_RUN_BY_QID, total_run_rows, missing_output_qids = merge_per_query_outputs(eval_qids, RUN_PATH)
MONOT5_EVAL_QIDS = sorted(set(MONOT5_RUN_BY_QID.keys()) & set(qrels.keys()), key=qid_sort_key)
MONOT5_METRICS = compute_metrics(MONOT5_RUN_BY_QID, qrels, MONOT5_EVAL_QIDS, k=10)
write_preview_tsv(MONOT5_RUN_BY_QID, scoreable_candidates, qrels, PREVIEW_PATH, max_rows=50)

elapsed_sec = time.time() - started_at
pairs_per_sec = newly_scored_pairs / elapsed_sec if elapsed_sec > 0 and newly_scored_pairs else 0.0

metrics_payload = {
    "config": dict(CFG),
    "paths": {key: str(value) for key, value in PATHS.items()},
    "data": {
        **DATA_SUMMARY,
        "num_evaluated_qids": int(len(MONOT5_EVAL_QIDS)),
        "num_scored_pairs": int(total_run_rows),
        "num_skipped_qids_without_output": int(len(missing_output_qids)),
        "evaluated_qids_default": "qids with qrels and scoreable candidates",
    },
    "bm25_candidate_order": BM25_METRICS,
    "monot5_base": MONOT5_METRICS,
    "runtime": {
        "device": str(DEVICE),
        "dtype": str(MODEL_DTYPE),
        "elapsed_sec": float(elapsed_sec),
        "newly_scored_pairs_this_run": int(newly_scored_pairs),
        "pairs_per_sec": float(pairs_per_sec),
        "progress_path": str(PROGRESS_PATH),
    },
}

save_json(dict(CFG), CONFIG_PATH)
save_json(metrics_payload, METRICS_PATH)

print("Full evaluation cell complete.")
print(f"Total qids evaluated: {len(MONOT5_EVAL_QIDS)}")
print(f"Total pairs in final run: {total_run_rows}")
print(f"Elapsed this run: {elapsed_sec:.2f} sec")
print(f"New scoring throughput: {pairs_per_sec:.2f} pairs/sec")
print("Progress:", json.dumps(progress_payload, indent=2))
print("monoT5 metrics:")
print(json.dumps(MONOT5_METRICS, indent=2))
print("Output files:")
for path in [RUN_PATH, METRICS_PATH, CONFIG_PATH, PROGRESS_PATH, MISSING_PATH, PREVIEW_PATH]:
    print(" ", path)


In [ ]:
if not METRICS_PATH.exists():
    raise FileNotFoundError(f"Metrics file has not been written yet: {METRICS_PATH}")

saved_metrics = json.loads(METRICS_PATH.read_text(encoding="utf-8"))
print(json.dumps(saved_metrics, indent=2, ensure_ascii=False))

print("\nArtifacts:")
print("Run file:", RUN_PATH)
print("Metrics JSON:", METRICS_PATH)
print("Config JSON:", CONFIG_PATH)
print("Progress JSON:", PROGRESS_PATH)
print("Missing-data JSON:", MISSING_PATH)
print("Preview TSV:", PREVIEW_PATH)

if RUN_PATH.exists():
    print("\nFirst run lines:")
    with RUN_PATH.open("r", encoding="utf-8") as handle:
        for idx, line in enumerate(handle):
            print(line.rstrip("\n"))
            if idx >= 4:
                break
